In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("GoogleBooksNgrams").getOrCreate()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1773609912422_0002,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
df = spark.read.csv("/user/hadoop/eng_1M_1gram/eng_1M_1gram.csv", header=True, inferSchema=True)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
df.show(5, truncate=False)
print("Columns:", df.columns)
print("Row count:", df.count())
print("Column count:", len(df.columns))
df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------+----+---------+-----+-----+
|token    |year|frequency|pages|books|
+---------+----+---------+-----+-----+
|inGermany|1927|2        |2    |2    |
|inGermany|1929|1        |1    |1    |
|inGermany|1930|1        |1    |1    |
|inGermany|1933|1        |1    |1    |
|inGermany|1934|1        |1    |1    |
+---------+----+---------+-----+-----+
only showing top 5 rows

Columns: ['token', 'year', 'frequency', 'pages', 'books']
Row count: 261823225
Column count: 5
root
 |-- token: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- frequency: integer (nullable = true)
 |-- pages: integer (nullable = true)
 |-- books: integer (nullable = true)

The dataset contains 261,823,225 rows and 5 columns: token, year, frequency, pages, and books. The year values range from 1520 to 2008. It's a very large dataset that contains a small number of null values in the numeric columns.

In [4]:
#describing dataset
df.describe().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+---------+------------------+-----------------+------------------+-----------------+
|summary|    token|              year|        frequency|             pages|            books|
+-------+---------+------------------+-----------------+------------------+-----------------+
|  count|261823225|         261822793|        261822793|         261822793|        261822793|
|   mean|      NaN|1910.5024307452102|396.6344411924442|210.14619198948046| 33.4816582947383|
| stddev|      NaN| 66.05224930213528|47883.07879567001| 7149.963110116652|233.7042538654384|
|    min|        !|              1520|                1|                 1|                1|
|    max|       ，|              2008|         43571378|           2035864|             6174|
+-------+---------+------------------+-----------------+------------------+-----------------+

In [5]:
df.createOrReplaceTempView("ngrams")

df_data = spark.sql("""
    SELECT *
    FROM ngrams
    WHERE token = 'data'
""")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
#4b
print("Filtered row count:", df_data.count())
df_data.printSchema()
df_data.orderBy("year").show(20, truncate=False)
df_data.describe().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Filtered row count: 316
root
 |-- token: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- frequency: integer (nullable = true)
 |-- pages: integer (nullable = true)
 |-- books: integer (nullable = true)

+-----+----+---------+-----+-----+
|token|year|frequency|pages|books|
+-----+----+---------+-----+-----+
|data |1584|16       |14   |1    |
|data |1614|3        |2    |1    |
|data |1627|1        |1    |1    |
|data |1631|22       |18   |1    |
|data |1637|1        |1    |1    |
|data |1638|2        |2    |1    |
|data |1640|1        |1    |1    |
|data |1642|1        |1    |1    |
|data |1644|4        |4    |1    |
|data |1647|1        |1    |1    |
|data |1651|1        |1    |1    |
|data |1674|1        |1    |1    |
|data |1690|1        |1    |1    |
|data |1693|1        |1    |1    |
|data |1697|1        |1    |1    |
|data |1699|1        |1    |1    |
|data |1700|1        |1    |1    |
|data |1701|11       |11   |2    |
|data |1702|1        |1    |1    |
|data |1

In [7]:
output_path = "/user/hadoop/eng_1M_1gram_filtered_data"
df_data.write.mode("overwrite").csv(output_path, header=True)
print("Done writing to:", output_path)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Done writing to: /user/hadoop/eng_1M_1gram_filtered_data